# RAG Pipeline - Exp3

* About
  * More modularized pipeline works for different data input
  * Deeper evaluation in retrieval and answer generation
* Learnings & Observations 🍀
  * Human labeled referenced documents referenced answers can have problems or go out-of-date soon, therefore
    * auto eval better not to assume every record has ground truth
    * in the eval metrics better to include the score than considers AI's output is better than the reference to the query

In [1]:
%load_ext autoreload
%autoreload 2

import os
import yaml
os.environ.pop("OPENAI_API_KEY", None)

from datasets import load_dataset
from llama_index.core import Document
import pandas as pd
import json

from utils import *

import warnings
warnings.filterwarnings('ignore')

resource module not available on Windows


### Load Data

* This applies to any dataset that has:
  * `contexts`: ground truth referenced documents, list format
  * `ground_truths`, list format
  * `question`

In [2]:
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")['baseline']

for item in fiqa_eval:
    if item['question'] == 'Have plenty of cash flow but bad credit':
        print(''.join(item['contexts']))
        print()
        print(''.join(item['ground_truths']))

This is probably a good time to note that credit is not a liquid asset, and not an emergency fund. Credit can be revoked or denied at any time, and Murphy's law states that you may have issues with credit when everything else goes wrong too.Assuming that a person has good financial discipline and is generally responsible with spending, I think that having a few hundred or thousand dollars extra of available credit is usually worth more to that person for the choice/flexibility it provides in unforeseen circumstance, versus the relatively minor hit that could be taken to their credit score.With bad credit but good income, I would simply save a large down payment. You're much more likely to get a mortgage with 25% down and a history of recently saving that 25% to show.

Set up a meeting with the bank that handles your business checking account. Go there in person and bring your business statements: profit and loss, balance sheet, and a spreadsheet showing your historical cash flow. The g

In [2]:
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")['baseline']

output_dir = "fiqa_raw_text"
os.makedirs(output_dir, exist_ok=True)

rag_lst = []
documents = []  # to store documents for llamindex
for idx, record in enumerate(fiqa_eval):
    context = ''.join(record['contexts'])
    gt = ''.join(record['ground_truths'])
    if 'answer' in record.keys():
        ai0_answer = record['answer'].strip()
    else:
        ai0_answer = None

    with open(os.path.join(output_dir, f"{idx}.txt"), "w", encoding="utf-8") as f:
        f.write(context)
    rag_lst.append({
        'question': record['question'],
        'context': context,
        'context_ct': len(record['contexts']),
        'ground_truth': gt,
        'ai0_answer': ai0_answer
    })
    doc = Document(
        text=context,
        metadata={
            "doc_name": idx
        }
    )
    documents.append(doc)

rag_df = pd.DataFrame(rag_lst)
print(rag_df.shape, max(rag_df['context_ct']), min(rag_df['context_ct']))
print(len(documents))
rag_df.head()

(30, 5) 1 1
30


,question,context,context_ct,ground_truth,ai0_answer
0,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,1,Have the check reissued to the proper payee.Ju...,The best way to deposit a cheque issued to an ...
1,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,1,Sure you can. You can fill in whatever you wa...,"Yes, you can send a money order from USPS as a..."
2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,1,You're confusing a lot of things here. Company...,"Yes, it is possible to have one EIN doing busi..."
3,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,1,"""I'm afraid the great myth of limited liabilit...",Applying for and receiving business credit can...
4,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,1,You should probably consult an attorney. Howev...,If your employer has closed and you need to tr...


### RAG Pipeline

In [8]:
with open("all_rag_pipeline_config.yaml", "r", encoding="utf-8") as f:
     all_config= yaml.safe_load(f)

In [6]:
for i in range(len(all_config['embedding_models'])):
    embed_model_str = all_config['embedding_models'][i]
    indexing_storage_dir = all_config['indexing_storage_dirs'][i]
    print(f"Embedding model: {embed_model_str}")
    create_vector_index(documents, embed_model_str, indexing_storage_dir)

Embedding model: BAAI/bge-small-en-v1.5


Parsing nodes:   0%|          | 0/30 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/35 [00:00<?, ?it/s]

Index saved to ./storage_m1


In [8]:
retriever_params = all_config['retriever_params']
cfgs = []
for i in range(len(all_config['yaml_config_files'])):
    yaml_config_file = 'output/saved_configs/' + all_config['yaml_config_files'][i]
    cfgs.append(yaml_config_file)
    config = {
        'llm_model': all_config['llm_models'][i],
        'embedding_model': all_config['embedding_models'][i],
        'indexing_storage_dir': all_config['indexing_storage_dirs'][i],
        'output_file': all_config['output_files'][i],
        'retriever_params': retriever_params
    }

    with open(yaml_config_file, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)

In [11]:
max_worker = os.cpu_count() - 1

await run_all_in_processes(cfgs, rag_lst, documents, max_workers=max_worker)

### Evaluation

In [ ]:
from langchain_groq import ChatGroq
import nest_asyncio
nest_asyncio.apply()

with open('eval_prompts.yaml', 'r') as file:
    prompt_versions = yaml.safe_load(file)

eval_llm = ChatGroq(
    groq_api_key=os.environ["GROQ_TOKEN"],
    model_name="openai/gpt-oss-20b", 
    temperature=0.7
)

eval_input_lst = []
for config in all_config['yaml_config_files']:
    yaml_config_file = 'output/saved_configs/' + config
    with open(yaml_config_file, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    output_file = config['output_file']
    response_lst = json.load(open(output_file, "r", encoding="utf-8"))
    eval_input = get_eval_input(response_lst)
    eval_input = pd.merge(eval_input, rag_df[['question', 'context']], left_on='query', right_on='question')
    eval_input.drop(columns=['question'], inplace=True)
    eval_input_lst.append(eval_input)
    print(eval_input.shape)
    display(eval_input.head())

(30, 6)


,query,ai_answer,referenced_answer,retrieved_content,question,context
0,How to deposit a cheque issued to an associate...,To deposit a cheque issued to an associate in ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...
1,Can I send a money order from USPS as a business?,"Yes, you can fill in your business name and ad...",Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...
2,1 EIN doing business under multiple business n...,"In some cases, a single EIN can be used for a ...",You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...
3,Applying for and receiving business credit,Establishing a relationship with your banker a...,"""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...
4,401k Transfer After Business Closure,If your employer's 401(k) plan is terminated d...,You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...


In [10]:
for idx, eval_input in enumerate(eval_input_lst):
    retrieval_error_records = eval_input[eval_input['retrieved_content'] != eval_input['context']]
    retrieval_quality = asyncio.run(get_retrieval_quality_output_async(eval_input, eval_llm,
                                                            prompt_versions['rq_prompt_template']))
    retrieval_quality.to_excel(all_config['eval_rq_output'][idx], index=False)
    print(retrieval_error_records.shape)
    display(retrieval_quality.head())

(7, 6)


,query,ai_answer,referenced_answer,retrieved_content,question,context,retrieval_quality_score,rq_reasoning
0,How to deposit a cheque issued to an associate...,To deposit a cheque issued to an associate in ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,3,The retrieved content exactly matches the cont...
1,Can I send a money order from USPS as a business?,"Yes, you can fill in your business name and ad...",Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,3,The retrieved content directly addresses the u...
2,1 EIN doing business under multiple business n...,"In some cases, a single EIN can be used for a ...",You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,3,The retrieved content is directly the same as ...
3,Applying for and receiving business credit,Establishing a relationship with your banker a...,"""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,3,The RETRIEVED CONTENT matches the context exac...
4,401k Transfer After Business Closure,If your employer's 401(k) plan is terminated d...,You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,3,The retrieved content mirrors the context text...


In [ ]:
ac_lst = []
for eval_input in eval_input_lst:
    answer_quality = asyncio.run(get_answer_quality_output_async(eval_input, eval_llm,
                                                            prompt_versions['aq_prompt_template']))
    ac_lst.append(answer_quality)
    print(answer_quality.shape)
    print(answer_quality['answer_quality_score'].value_counts(normalize=True))
    display(answer_quality.head())

(30, 6)
answer_accuracy_score
2    0.366667
3    0.300000
1    0.233333
0    0.100000
Name: proportion, dtype: float64


,query,ai_answer,referenced_answer,retrieved_content,answer_accuracy_score,ac_reasoning
0,How to deposit a cheque issued to an associate...,To deposit a cheque issued to an associate in ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,2,Answer is relevant and gives correct step of s...
1,Can I send a money order from USPS as a business?,"Yes, you can fill in your business name and ad...",Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,2,AI's answer correctly states you can use a bus...
2,1 EIN doing business under multiple business n...,"In some cases, a single EIN can be used for a ...",You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,1,AI’s answer is relevant but contradicts the re...
3,Applying for and receiving business credit,Establishing a relationship with your banker a...,"""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,2,AI answer is relevant and covers most key poin...
4,401k Transfer After Business Closure,If your employer's 401(k) plan is terminated d...,You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,1,"Answer mentions IRA consolidation, but referen..."
